<a href="https://colab.research.google.com/github/hottp2988/tibame-20251203/blob/main/%E7%B4%85%E9%85%92_DT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 安裝與匯入 (Colab 必備)
!pip install tensorflow_decision_forests
import tensorflow_decision_forests as tfdf
import pandas as pd
import numpy as np

# 2. 載入紅酒資料並建立資料集
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')

# TF-DF 喜歡用整數當標籤，我們確保 quality 是 int
df["quality"] = df["quality"].astype(int)

# 3. 拆分資料 (75:25)
train_df = df.sample(frac=0.75, random_state=42)
test_df = df.drop(train_df.index)

# 4. 轉換成 TensorFlow Dataset 格式 (TF-DF 的特殊要求)
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train_df, label="quality")
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(test_df, label="quality")

# 5. 建立 CartModel
# 你可以在這裡設定樹的深度 (max_depth)
model = tfdf.keras.CartModel(max_depth=5)

# 6. 開始訓練
# 雖然 CartModel 是一次算完，但 TF-DF 會列出它處理特徵的進度
model.fit(train_ds)

# 7. 查看訓練後的「詳細結果報告」
# 31
inspector = model.make_inspector()

# 32 顯示模型類型
print(f"模型類型: {inspector.model_type()}")

# 33 修正這裡：改用通用方法來抓節點數量
# 有些 inspector 版本不支援直接讀取屬性，我們改從模型規格裡抓
print(f"訓練節點數量 (Nodes): {len(inspector.features())}") # 顯示使用的特徵數
print(f"樹的數量 (Trees): {inspector.num_trees()}") # CartModel 應該是 1

# 35 顯示特徵重要性 (這個通常最重要)
print("\n--- 特徵重要性排名 (Importance) ---")
# TF-DF 的重要性儲存在字典裡，我們抓 SUM_SCORE
importances = inspector.variable_importances()
if "SUM_SCORE" in importances:
    for val in importances["SUM_SCORE"]:
        print(f"特徵: {val[0].name}, 分數: {val[1]:.4f}")
else:
    # 如果沒有 SUM_SCORE，就印出第一個可用的指標
    first_key = list(importances.keys())[0]
    for val in importances[first_key]:
        print(f"特徵: {val[0].name}, 指標({first_key}): {val[1]:.4f}")


Use /tmp/tmpfhkii_k6 as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.234444. Found 1199 examples.
Training model...
Model trained in 0:00:00.029631
Compiling model...
Model compiled.
模型類型: RANDOM_FOREST
訓練節點數量 (Nodes): 11
樹的數量 (Trees): 1

--- 特徵重要性排名 (Importance) ---
特徵: alcohol, 分數: 125.8187


In [ ]:
tfdf.model_plotter.plot_model_in_colab(model, max_depth=3)

In [ ]:
# 1. 取得模型的 inspector
inspector = model.make_inspector()

# 2. 顯示訓練日誌 (Training Logs)
# 對於 CartModel，這會顯示它在訓練時處理了多少樣本、特徵
print("--- CartModel 訓練日誌 ---")
print(inspector.training_logs())

# 3. 顯示模型是如何「一步步」蓋起來的 (Model Description)
# 這會列出每一層的判斷邏輯
print("\n--- 決策樹詳細結構清單 ---")
print(model.summary())

--- CartModel 訓練日誌 ---
[TrainLog(num_trees=1, evaluation=Evaluation(num_examples=92, accuracy=0.6195652173913043, loss=1.0569402277469635, rmse=None, ndcg=None, aucs=[0.0, 0.0, 0.0, 0.0, 0.0, 0.52183908045977, 0.7423809523809524, 0.6486215538847118, 0.7142857142857143, 0.8277777777777777], auuc=None, qini=None))]

--- 決策樹詳細結構清單 ---
Model: "cart_model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
Total params: 1 (1.00 Byte)
Trainable params: 0 (0.00 Byte)
Non-trainable params: 1 (1.00 Byte)
_________________________________________________________________
Type: "RANDOM_FOREST"
Task: CLASSIFICATION
Label: "__LABEL"

Input Features (11):
	alcohol
	chlorides
	citric_acid
	density
	fixed_acidity
	free_sulfur_dioxide
	pH
	residual_sugar
	sulphates
	total_sulfur_dioxide
	volatile_acidity

No weights

Variable Importance: INV_MEAN_MIN_DEPTH:
    1. "alcohol"  1.000000 

Variable Importance: NUM_AS_ROOT:
  